In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.11 Bound States in One Dimension

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.11",
    title="Bound States in One Dimension",
    blurb="Trapping quantizes. A particle confined to a well has only a discrete "
    "ladder of allowed energies, and our grid solver finds them for any well shape. "
    "Symmetric wells sort their states by parity; deeper wells hold more of them; "
    "and two wells side by side reveal the most consequential structure of all — a "
    "pair of tunnelling-split levels that is, exactly, the two-state system we began "
    "the quantum story with.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

In [§6.10](schrodinger-on-a-computer.ipynb) we built an engine: hand it a potential on a grid and `numpy.linalg.eigh` returns the
spectrum and the stationary states. This notebook *uses* that engine — we do not rebuild it — to do
physics that pencil and paper struggle with, and it ends by reconnecting wave mechanics to the qubit
that opened the volume.

The thread is **confinement quantizes**. A particle trapped in an attractive well has a discrete
ladder of **bound states** (normalizable, with energy below the potential far away), and the grid
method finds them for any well shape. We start with the **finite square well**, whose exact spectrum
is set by a *transcendental* matching condition that has no closed-form solution — and watch the grid
reproduce it without our doing any matching at all, a vivid demonstration of why the computational
approach is liberating. A symmetric well brings in **parity**: because the Hamiltonian commutes with
the reflection operator, every bound state is cleanly even or odd, and the parities alternate up the
ladder — the first concrete case of a symmetry labelling states. We count how many states a well
holds and find the one-dimensional rule that an attractive well *always* binds at least one.

Then the payoff. Put two wells side by side, and each single-well level **splits** into a
near-degenerate **doublet** — a symmetric state just below, an antisymmetric one just above —
separated by an energy that comes entirely from **tunnelling** through the central barrier and is
exponentially sensitive to its width. The localized combinations are not energy eigenstates, so a
particle placed in one well **sloshes** to the other and back. This doublet is not a curiosity: it is
the ammonia molecule's nitrogen inversion (the ammonia maser), and it is the physical origin of the
**qubit** of [§6.4](stern-gerlach-qubit.ipynb). The abstract two-state system we began with turns out to be the most natural thing
in the world — put a particle between two wells, and quantum mechanics hands you a qubit.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the [§6.10](schrodinger-on-a-computer.ipynb) finite-difference solver (`numpy.linalg.eigh`),
`scipy.optimize.brentq` for the transcendental roots, and the parity diagnostic
`numpy.sum(psi*psi[::-1])*dx`.

> **Method reuse.** The `hamiltonian`/`solve` finite-difference eigensolver below is the one built and
> validated in [§6.10](schrodinger-on-a-computer.ipynb) (the $(1,-2,1)/dx^2$ kinetic stencil plus $\mathrm{diag}\,V$, diagonalized by
> `numpy.linalg.eigh`, eigenvectors normalized by $\sqrt{dx}$); we restate it for self-containment and
> then treat it purely as a tool — the new content here is the *physics*.
>
> **Conventions.** $\hbar=1$, $m=1$. A symmetric grid on $[-L/2,L/2]$ (so the parity
> diagnostic `psi[::-1]` is exactly $\psi(-x)$), with the box large enough that bound-state wave
> functions decay before the edges. The finite well is $V=-V_0$ for $|x|<a$; the double well is two
> such wells centred at $\pm x_c$. **Numerical honesty:** the tunnelling splitting falls exponentially
> with the barrier, and once it drops below the diagonalization floor ($\sim10^{-13}$) the computed
> doublet degeneracy is meaningless — we stay in the resolvable regime and say so (a [§6.10](schrodinger-on-a-computer.ipynb) callback).
> See Griffiths (the finite well, parity, the double well); the Feynman Lectures Vol. III (the ammonia
> maser); and Notebooks [§6.10](schrodinger-on-a-computer.ipynb) (the eigenmethod), [§6.2](operators-spectral-theorem.ipynb) (parity commutes with $H$), [§6.4](stern-gerlach-qubit.ipynb) (the two-state
> system), [§6.7](time-evolution.ipynb) (tunnelling oscillation = two-level beating).

## Theory in brief

### Bound states and the discrete spectrum

A particle in an attractive well has **bound states** — normalizable eigenstates with energy below
the potential at infinity,

```{math}
:label: eq-bound
\hat H\psi_n=E_n\psi_n,\qquad E_n<0\ \text{(for a well vanishing at infinity)},\qquad \int|\psi_n|^2dx<\infty ,
```

forming a **discrete** spectrum (confinement quantizes), unlike the continuous spectrum of free states
([§6.9](position-representation.ipynb)). The [§6.10](schrodinger-on-a-computer.ipynb) eigenmethod finds them: build $H$ on a large box, diagonalize, keep the $E<0$ levels.

### The finite square well and the transcendental condition

For $V=-V_0$ inside $|x|<a$ and $0$ outside, matching the interior oscillation ($k=\sqrt{2m(E+V_0)}/
\hbar$) to the exterior decay ($\kappa=\sqrt{-2mE}/\hbar$) at the walls gives the **transcendental**
conditions

```{math}
:label: eq-finite-well
k\tan(ka)=\kappa\ \text{(even)},\qquad -k\cot(ka)=\kappa\ \text{(odd)} ,
```

solvable only numerically (`scipy.optimize.brentq`). The grid method gives the *same* energies without
doing the matching. Unlike the infinite well, the finite well has **finitely** many bound states, and
the wave functions **leak** into the classically forbidden region (exponential tails — a tunnelling
preview).

### Parity

If $V(-x)=V(x)$, then $H$ commutes with the **parity** operator $P\psi(x)=\psi(-x)$, so ([§6.2](operators-spectral-theorem.ipynb)) they
share eigenstates: every bound state has **definite parity**,

```{math}
:label: eq-parity
[H,P]=0\ \Longrightarrow\ \psi_n(-x)=\pm\psi_n(x),\qquad \text{ground state even, parities alternate} .
```

Parity is a conserved quantum number that halves the problem — the first concrete symmetry-labels-states
example (the theme of complete sets of commuting observables, [§6.6](pauli-uncertainty.ipynb)).

### Bound-state counting

The number of bound states grows with depth and width; counting the crossings of the transcendental
conditions above gives

```{math}
:label: eq-counting
N_{\text{bound}}\approx\Big\lfloor\frac{2a}{\pi}\frac{\sqrt{2mV_0}}{\hbar}\Big\rfloor+1 ,
```

and a one-dimensional attractive well **always** binds at least one state, however shallow (unlike
3D). Each new state enters at threshold ($E\to0$) as the well deepens.

### The double well and tunnelling doublets

Two wells separated by a barrier: each single-well level splits into a near-degenerate **doublet**, a
symmetric (even) state just below and an antisymmetric (odd) state just above, split by $\Delta E$,

```{math}
:label: eq-double-well
|L\rangle,|R\rangle=\tfrac{1}{\sqrt2}(|\text{sym}\rangle\pm|\text{antisym}\rangle),\qquad T=\frac{2\pi\hbar}{\Delta E},\qquad \Delta E\ \text{exponentially small in the barrier} .
```

The localized $|L\rangle,|R\rangle$ are not energy eigenstates, so a particle in one well **tunnels**
to the other with period $T=2\pi\hbar/\Delta E$ (the [§6.7](time-evolution.ipynb) beating of two energy eigenstates). This is
the ammonia molecule and the physical origin of the **qubit** of [§6.4](stern-gerlach-qubit.ipynb): the symmetric/antisymmetric
doublet is the qubit's $|0\rangle/|1\rangle$, the tunnelling splitting its energy gap.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.optimize import brentq

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

HBAR = 1.0  # ℏ = 1
MASS = 1.0  # m = 1

# A symmetric grid on [−L/2, L/2] so that psi[::-1] is exactly ψ(−x) (the parity diagnostic).
L_BOX = 20.0
N_GRID = 1400
X = np.linspace(-L_BOX / 2, L_BOX / 2, N_GRID)
DX = X[1] - X[0]


def hamiltonian(x, V):
    """The §6.10 finite-difference Hamiltonian: $-(\\hbar^2/2m)(1,-2,1)/dx^2$ stencil plus $\\mathrm{diag}\\,V$.

    Restated from §6.10 for self-containment (assembled with ``numpy.diag``); a real symmetric matrix.
    """
    n = len(x)
    dx = x[1] - x[0]
    lap = (
        np.diag(-2 * np.ones(n))
        + np.diag(np.ones(n - 1), 1)
        + np.diag(np.ones(n - 1), -1)
    ) / dx**2
    return -(HBAR**2 / (2 * MASS)) * lap + np.diag(V)


def solve(x, V):
    """Diagonalize: energies and normalized stationary states (the §6.10 eigenmethod, reused) {eq}`eq-bound`.

    ``numpy.linalg.eigh`` returns sorted real eigenvalues and orthonormal eigenvectors; columns are
    divided by $\\sqrt{dx}$ so $\\int|\\psi_n|^2dx=1$. We *use* this as a tool here; it was built and
    validated in §6.10.
    """
    energies, states = np.linalg.eigh(hamiltonian(x, V))
    return energies, states / np.sqrt(x[1] - x[0])


def parity(psi, dx):
    """The parity diagnostic $\\int\\psi(x)\\psi(-x)\\,dx$ {eq}`eq-parity`.

    On a grid symmetric about the origin, $\\psi(-x)$ is the reversed array ``psi[::-1]``, so this is
    ``numpy.sum(psi*psi[::-1])*dx``. For a normalized definite-parity state it returns $+1$ (even) or
    $-1$ (odd).
    """
    return np.sum(psi * psi[::-1]) * dx


def count_bound(energies):
    """The number of bound states — the count of negative eigenvalues {eq}`eq-counting`."""
    return int(np.sum(energies < 0))


def localized_states(sym, antisym):
    """The localized combinations $|L\\rangle,|R\\rangle=(|\\text{sym}\\rangle\\pm|\\text{antisym}\\rangle)/\\sqrt2$ {eq}`eq-double-well`.

    Two equal superpositions of the symmetric and antisymmetric doublet members; each is concentrated in
    one well. They are *not* energy eigenstates, so they tunnel.
    """
    return (sym + antisym) / np.sqrt(2), (sym - antisym) / np.sqrt(2)


def transcendental_roots(V0, a):
    """The finite-well bound energies from the transcendental conditions {eq}`eq-finite-well`.

    Solves $k\\tan(ka)=\\kappa$ (even) and $-k\\cot(ka)=\\kappa$ (odd) branch by branch in the
    dimensionless $u=ka$ (so each `scipy.optimize.brentq` bracket avoids the $\\tan/\\cot$ poles), with
    $R=\\sqrt{2mV_0a^2}/\\hbar$ the well-strength parameter. Returns the sorted bound energies.
    """
    R = np.sqrt(2 * MASS * V0 * a**2) / HBAR
    roots_u = []
    n = 0  # even branches: u tan u = sqrt(R²−u²) on (nπ, min((n+½)π, R))
    while n * np.pi < R:
        lo, hi = n * np.pi + 1e-6, min((n + 0.5) * np.pi - 1e-6, R - 1e-9)
        if lo < hi:
            f = lambda u: u * np.tan(u) - np.sqrt(max(R**2 - u**2, 1e-30))
            if f(lo) * f(hi) < 0:
                roots_u.append(brentq(f, lo, hi))
        n += 1
    n = 0  # odd branches: −u cot u = sqrt(R²−u²) on ((n+½)π, min((n+1)π, R))
    while (n + 0.5) * np.pi < R:
        lo, hi = (n + 0.5) * np.pi + 1e-6, min((n + 1) * np.pi - 1e-6, R - 1e-9)
        if lo < hi:
            f = lambda u: -u / np.tan(u) - np.sqrt(max(R**2 - u**2, 1e-30))
            if f(lo) * f(hi) < 0:
                roots_u.append(brentq(f, lo, hi))
        n += 1
    return np.sort([(HBAR**2 / (2 * MASS)) * (u / a) ** 2 - V0 for u in roots_u])

## Exercise 1 — The finite square well: spectrum and leaking wave functions

Solve a finite square well $V=-V_0$ for $|x|<a$ (and $0$ outside) and find its bound states.
Report the bound energies and observe that the eigenfunctions leak into the classically forbidden
region with exponential tails {eq}`eq-bound`, {eq}`eq-finite-well`.

1. Build $V=-V_0$ for $|x|<a$ on the large box `X` with `numpy.where`.
2. Solve with the [§6.10](schrodinger-on-a-computer.ipynb) `solve` helper (`numpy.linalg.eigh`) and keep the $E<0$ eigenvalues — the
   bound states.
3. Report how many there are and their energies (finitely many, unlike the infinite well).
4. Plot the eigenfunctions and note the exponential tails extending past $|x|=a$ into the
   forbidden region — the wave function penetrates the wall, a tunnelling preview.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    0 < n_bound < N_GRID and np.all(E_bound < 0) and tail_prob > 0,
    "a finite well binds a discrete, finite set of states (E<0) whose wave functions leak into the classically forbidden region",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The transcendental condition, checked against the grid

Solve the finite well's exact transcendental matching condition and confirm the grid eigenvalues
of Exercise 1 reproduce it — the grid gets the same answer without our doing the matching
{eq}`eq-finite-well`.

1. Write the matching conditions $k\tan(ka)=\kappa$ (even) and $-k\cot(ka)=\kappa$ (odd), with
   $k=\sqrt{2m(E+V_0)}/\hbar$ and $\kappa=\sqrt{-2mE}/\hbar$.
2. Find their roots in $(-V_0,0)$ with `scipy.optimize.brentq`, searching branch by branch in
   $u=ka$ so each bracket avoids the $\tan/\cot$ poles (the `transcendental_roots` helper).
3. Compare to the grid spectrum from Exercise 1.
4. Confirm they agree — the deepest levels to $\sim10^{-3}$, the shallowest (most diffuse, and
   sitting at the well's discontinuous edge) to $\sim10^{-2}$, the resolution caveat of [§6.10](schrodinger-on-a-computer.ipynb).
   Frame: the grid method is liberating — same answer, no transcendental algebra.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.sort(E_bound),
    roots,
    "the grid spectrum matches the finite well's transcendental matching condition (without our doing the matching)",
    atol=5e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Parity

Show that the bound states of the symmetric finite well have definite parity, even or odd, with
the ground state even and the parities alternating up the ladder {eq}`eq-parity`.

1. Confirm the potential is symmetric, $V(-x)=V(x)$ (the grid is symmetric, so `numpy.allclose(V,
   V[::-1])`), which makes $[H,P]=0$.
2. For each bound state compute the parity diagnostic $\int\psi(x)\psi(-x)\,dx$ with the `parity`
   helper (`numpy.sum(psi*psi[::-1])*dx`).
3. Confirm each value is $+1$ (even) or $-1$ (odd).
4. Observe the ground state is even and the parities alternate $+,-,+,-,\dots$. Frame: a conserved
   quantum number from a symmetry ([§6.2](operators-spectral-theorem.ipynb) — $P$ and $H$ share eigenstates), which halves the problem.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
alternating = (
    np.allclose(np.abs(parities), 1.0, atol=1e-3)
    and parities[0] > 0
    and np.all(parities[:-1] * parities[1:] < 0)
)
validate.check(
    symmetric and alternating,
    "a symmetric well gives definite-parity eigenstates (±1), even ground state, parities alternating up the ladder",
)

## Exercise 4 — Bound-state counting

Determine how the number of bound states depends on the well depth $V_0$, compare to the estimate
$\lfloor(2a/\pi)\sqrt{2mV_0}/\hbar\rfloor+1$, and confirm the one-dimensional rule that even a
shallow well binds at least one state {eq}`eq-counting`.

1. Solve the well for a range of depths $V_0$ (the `solve` helper at each).
2. Count the $E<0$ states with `count_bound`.
3. Compare to the estimate $\lfloor(2a/\pi)\sqrt{2mV_0}/\hbar \rfloor+1$.
4. Confirm the count rises in steps as $V_0$ grows (a new state enters at threshold), and that
   even the shallowest well has $\ge1$ bound state. Frame: deeper and wider wells hold more; a 1-D
   attractive well always binds.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    counts[0] >= 1
    and np.all(np.diff(counts) >= 0)
    and counts[-1] > counts[0]
    and np.all(np.abs(counts - np.array(estimates)) <= 1),
    "the bound-state count grows with depth (matching the threshold estimate) and is ≥1 even for a shallow well",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The double well and tunnelling doublets

Solve a symmetric **double well** — two wells separated by a barrier — and analyze its lowest
doublet: confirm the two lowest states are a near-degenerate pair, a symmetric (even) state just
below an antisymmetric (odd) one, and that their $\pm$ combinations localize in the two wells
{eq}`eq-double-well`.

1. Build two wells of depth $V_0$ and half-width $a$ centred at $\pm x_c$ with `numpy.where`.
2. Solve with the `solve` helper.
3. Identify the lowest two states as a doublet: confirm their parities (even below, odd above, via
   the `parity` helper) and measure the splitting $\Delta E=E_1-E_0$ — tiny compared with the gap
   to the next level.
4. Form the localized combinations
   $|L\rangle,|R\rangle=(|\text{sym}\rangle\pm|\text{antisym}\rangle)/\sqrt2$ with
   `localized_states` and show each is concentrated in one well. Frame: tunnelling splits each
   single-well level into a doublet.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    parity(sym, DX) > 0
    and parity(antisym, DX) < 0
    and 0 < delta_E < 0.1 * (E_dw[2] - E_dw[1])
    and np.sum(left[X < 0] ** 2) * DX > 0.95
    and np.sum(right[X > 0] ** 2) * DX > 0.95,
    "tunnelling splits the level into a symmetric/antisymmetric doublet, whose ± combinations localize in the two wells",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Tunnelling oscillation and the qubit connection

Show that a particle localized in one well tunnels to the other and back with period
$T=2\pi\hbar/\Delta E$, and identify this two-state system with the qubit of [§6.4](stern-gerlach-qubit.ipynb)
{eq}`eq-double-well`.

1. Start in $|L\rangle$ (not an energy eigenstate, but $(|\text{sym}\rangle+|\text{
   antisym}\rangle)/\sqrt2$).
2. Evolve it ([§6.7](time-evolution.ipynb)): each doublet member carries its phase $e^{-iE_nt/ \hbar}$, so
   $|\psi(t)\rangle=(e^{-iE_0t}|\text{sym}\rangle+e^{-iE_1t}|\text{antisym}\rangle)/\sqrt2$.
3. Compute the probability in the right well versus time and confirm it oscillates with period
   $T=2\pi\hbar/\Delta E$ (the two energies beat — [§6.7](time-evolution.ipynb)).
4. Identify the symmetric/antisymmetric doublet as the qubit's $|0\rangle/|1\rangle$ and $\Delta
   E$ as its gap: this is the ammonia molecule's inversion and the physical origin of the [§6.4](stern-gerlach-qubit.ipynb)
   qubit. Frame: the abstract two-state system, derived from a potential.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [P_right[0], P_right[len(times) // 2], P_right[-1]],
    [0.0, 1.0, 0.0],
    "a particle localized in one well tunnels fully to the other and back with period 2πℏ/ΔE — the tunnelling doublet is a two-state system",
    atol=0.05,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Splitting versus barrier, and where the numerics would run out *(student)*

Study how the doublet splitting $\Delta E$ depends on the barrier width (the well separation), and
mark the floor below which the computation would stop being trustworthy {eq}`eq-double-well`.

1. Compute $\Delta E$ for a series of separations $x_c$ (wider $x_c$ = wider barrier), each via
   the `solve` helper.
2. Plot $\Delta E$ on a log scale against $x_c$ and observe the **exponential** decrease (a
   straight line on the log plot — tunnelling is exponentially sensitive to the barrier).
3. Mark the diagonalization floor ($\sim10^{-13}$) below which the computed doublet degeneracy
   would become meaningless, and confirm the scan stays above it — in the resolvable regime.
4. State the lesson: the exponential trend is physics; the floor is where the numerics would run
   out, and stopping above it is honest computation (a [§6.10](schrodinger-on-a-computer.ipynb) callback). Frame: tunnelling — and the limit of our precision — are both
   exponential in the barrier.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    splittings[0] > 1e-3
    and splittings[-1] < 1e-8
    and slope < 0
    and np.all(np.diff(splittings[resolvable]) < 0),
    "the tunnelling splitting decreases exponentially with barrier width, staying above the numerical floor (~1e-13) below which the numerics would run out",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Confinement, symmetry, and the birth of the two-state system

Trapping a particle quantizes its energy into a discrete ladder; a symmetric well sorts that ladder by
**parity**; deeper wells hold more rungs, and in one dimension even the shallowest well holds at least
one. The finite well showed the spectrum is set by a transcendental condition the grid solves for free,
and that bound states **leak** past their walls. And when two wells sit side by side, that leakage
becomes **tunnelling**: each rung splits into a doublet whose two members beat against each other, a
particle sloshing from well to well with period $2\pi\hbar/\Delta E$.

**This exercise (synthesis).** No new computation: the connection is the result. That doublet is not a
curiosity — it is the ammonia molecule's inversion, and it is the **qubit** we began the quantum story
with in [§6.4](stern-gerlach-qubit.ipynb), now *derived* from a potential rather than postulated. We opened this volume with an
abstract two-state system and called it the simplest quantum object; here it turns out to be the most
natural one, handed to us by quantum mechanics the moment we place a particle between two wells. The
next notebook ([§6.12](harmonic-oscillator.ipynb)) solves the single most important bound-state problem of all — the **harmonic
oscillator**, the universal description of any system near a minimum — both by diagonalizing on the
grid and by the elegant algebra of ladder operators.

## Notebook summary

The [§6.10](schrodinger-on-a-computer.ipynb) eigenmethod applied to bound-state physics — and the qubit, derived.

- **Bound states** {eq}`eq-bound`: confinement gives a discrete ladder of normalizable $E<0$ states;
  the [§6.10](schrodinger-on-a-computer.ipynb) solver finds them for any well.
- **The finite well** {eq}`eq-finite-well`: finitely many states, set by a transcendental condition
  (`scipy.optimize.brentq`, branch by branch) that the grid reproduces without matching; wave
  functions leak past the walls.
- **Parity** {eq}`eq-parity`: a symmetric well gives definite-parity states ($\int\psi(x)\psi(-x)dx=
  \pm1$), even ground state, alternating — a symmetry labelling states ($[H,P]=0$, [§6.2](operators-spectral-theorem.ipynb)).
- **Counting** {eq}`eq-counting`: the number of states grows with depth ($\approx\lfloor(2a/\pi)
  \sqrt{2mV_0}/\hbar\rfloor+1$); a 1-D well always binds $\ge1$.
- **The double well** {eq}`eq-double-well`: tunnelling splits each level into a symmetric/antisymmetric
  doublet; the localized $(|\text{sym}\rangle\pm|\text{antisym}\rangle)/\sqrt2$ tunnel with period
  $2\pi\hbar/\Delta E$; $\Delta E$ is exponentially small in the barrier (until the $\sim10^{-13}$
  numerical floor).

This doublet is the ammonia molecule and the physical origin of the [§6.4](stern-gerlach-qubit.ipynb) qubit: the two-state system
we postulated, now derived from a potential.

## Outlook

- **The harmonic oscillator ([§6.12](harmonic-oscillator.ipynb))**: ladder operators and the analytic solution, checked against the
  grid; coherent states.
- **Scattering and tunnelling through barriers ([§6.13](scattering-tunneling.ipynb))**: transmission, resonances, wave-packet
  dynamics.
- **Identical particles ([§6.20](identical-particles.ipynb))**: the same symmetric/antisymmetric structure in a new role.
- **Molecular physics**: tunnelling, inversion, and the ammonia maser (a horizon).
- **Cross-reference** [§6.10](schrodinger-on-a-computer.ipynb) (the eigenmethod), [§6.2](operators-spectral-theorem.ipynb) (parity / commuting operators), [§6.4](stern-gerlach-qubit.ipynb) (the two-state
  system), [§6.7](time-evolution.ipynb) (tunnelling oscillation = two-level beating), and forward to [§6.12](harmonic-oscillator.ipynb), [§6.13](scattering-tunneling.ipynb), [§6.20](identical-particles.ipynb).

In [ ]:
from ecp.style import footer

footer()